In [2]:
import numpy as np
import sys
sys.path.append("..") 
import utils.funciones as fm
from IPython.display import clear_output
np.set_printoptions(precision=20, suppress=False)

**DATOS INICIALES**

In [3]:
import numpy as np

# HORIZONS
# VARIABLES
user_float = np.float64
G_const = 4*(np.pi)**2  #(AU^3/(M_sun*year^2)))
dt_const = 0.0005
mass = np.array([
    1988410E24,     #Sun
    3.302E23,       #Mercury
    48.685E23,      #Venus
    5.97219E24,     #Earth
    6.4171E23,      #Mars
    18.9819E26,     #Jupiter
    5.6834E26,      #Saturn
    86.813E24,      #Uranus
    102.409E24,     #Neptune
    1.303E22        #Pluto
    ], dtype=user_float)/1988410E24

tropical_year = 365.242190419
print("Year in days:", tropical_year)

#AU
position_t0 = np.array([
    [-5.730606324378153E-03, -4.910557082772070E-03,  1.795094577858112E-04], #Sun   
    [-3.930336148500469E-01, -1.666347517169735E-01,  2.248723485902817E-02], #Mercury   
    [4.476881591494200E-01,  5.573055222132830E-01, -1.826185079464067E-02], #Venus
    [-1.844140472974828E-01,  9.620722382946830E-01,  1.284152186349546E-04], #Earth
    [-5.274164728925163E-01,  1.520324019719684E+00,  4.493510749539490E-02], #Mars
    [1.050302939252324E+00,  4.966541604941111E+00, -4.409855432676144E-02], #Jupiter
    [9.455336665176439E+00, -1.769525277925947E+00, -3.456968909591847E-01], #Saturn
    [1.109789820880128E+01,  1.608957335509925E+01, -8.401882312746477E-02], #Uranus
    [2.987419674943718E+01, -6.390985521271113E-01, -6.753202855837556E-01], #Neptune
    [1.822309610897269E+01, -3.001291441773554E+01, -2.059639830476407E+00], #Pluto
], dtype=user_float)

#AU/day
velocity_t0 = np.array([
    [7.160788611314778E-06, -3.666546272590425E-06, -1.176972196034231E-07], #Sun   
    [5.031590985068973E-03, -2.474711985703500E-02, -2.483041558403887E-03], #Mercury   
    [-1.579911349143828E-02,  1.260639609552168E-02,  1.085100185838057E-03], #Venus  
    [-1.719757779305811E-02, -3.197199735579798E-03, -1.122400455364092E-07], #Earth
    [-1.270467411479119E-02, -3.342506132667855E-03,  2.416611203580803E-04], #Mars
    [-7.469111612367896E-03,  1.920799528808175E-03,  1.591398359241618E-04], #Jupiter
    [7.165415690664376E-04,  5.471430990255198E-03, -1.233522942159990E-04], #Saturn
    [-3.266567450208665E-03,  2.049861782622937E-03,  4.996819331008718E-05], #Uranus
    [4.657674111212642E-05,  3.156723229456242E-03, -6.648766294387875E-05], #Neptune
    [2.767433185219602E-03,  9.357142006899497E-04, -9.006318946387802E-04], #Pluto
], dtype=user_float)*tropical_year

#Linear Momentum
momentum_t0 = mass[:,None]*velocity_t0 # all and variables

Year in days: 365.242190419


# Physical Model Initialization: Center of Mass Corrections

For an $N$-body model to be physically coherent and numerically stable, we must ensure that the system does not have an artificial "drift". This is achieved by translating the system to the **Center of Mass Reference Frame**.

---

### 1. Position Correction
If the position is not corrected, the group of bodies might be very far from the origin $(0,0,0)$, which causes a loss of precision in floating-point calculations and makes visualization difficult.

We calculate the position of the center of mass $\vec{R}_{cm}$:
$$\vec{R}_{cm} = \frac{\sum_{i=1}^{n} m_i \vec{r}_i}{\sum_{i=1}^{n} m_i}$$

**Applied correction:**
$$\vec{r}_{i, \text{new}} = \vec{r}_i - \vec{R}_{cm}$$

> **Effect:** The system is perfectly centered at the origin, such that $\sum m_i \vec{r}_{i, \text{new}} = 0$.

---

### 2. Momentum Correction
A system without external forces must have a net zero momentum so it does not displace linearly through space. The initial net momentum is calculated as:
$$\vec{P}_{net} = \sum_{i=1}^{n} \vec{p}_i$$

To correct this without altering the relative physics between the bodies, we subtract from each particle a fraction of the total momentum proportional to its mass contribution:
$$\vec{p}_{i, \text{new}} = \vec{p}_i - \left( \frac{m_i}{M_{total}} \right) \vec{P}_{net}$$

**Why proportional to the mass?**
Because this is equivalent to subtracting the same drift velocity $\vec{V}_{cm}$ from all bodies:
$$\Delta \vec{v} = \frac{\vec{P}_{net}}{M_{total}}$$
This guarantees that the internal structure of the system (relative velocities) remains intact.

---

### 3. Verification of Equilibrium Conditions
After the corrections, the model must satisfy:
1. **Zero Net Momentum:** $\sum \vec{p}_i \approx 0$
2. **CM Position at the origin:** $\vec{R}_{cm} \approx 0$

**NET NET LINEAR MOMENTUM CORRECTION**

In [9]:
total_mass = mass.sum()
print("Total Mass:", total_mass)

net_momentum = momentum_t0.sum(axis=0)
print("Net Momentum:", net_momentum)

mass_proportion = mass/total_mass

momentum_t0 = momentum_t0 - net_momentum*mass_proportion[:,None]

print("Corrected Net Momentum:", momentum_t0.sum(axis=0))

Total Mass: 1.0013415631735911
Net Momentum: [ 5.9148376881216797e-20 -4.8076564382194521e-20  2.0083945272787512e-21]
Corrected Net Momentum: [ 5.9148376881216797e-20 -4.8076564382194521e-20  2.0083945272787512e-21]


**POSITION CORRECTION**

In [18]:
net_position = (mass[:, None] * position_t0).sum(axis=0)

r_cm = net_position / total_mass

print("Original Center of Mass:", r_cm)

position_t0 = position_t0 - r_cm

corrected_net_position = (mass[:, None] * position_t0).sum(axis=0)

print("Corrected Center of Mass:", corrected_net_position / total_mass)

Original Center of Mass: [-9.796557619458636e-20 -5.564423580399468e-20 -9.471745957966197e-21]
Corrected Center of Mass: [-9.796557619458636e-20 -5.564423580399468e-20 -9.471745957966197e-21]


***Yoshida 4to orden***

***[0.6756035959798289,-0.1756035959798288,-0.1756035959798288,0.6756035959798289]*** 

***[1.3512071919596578,-1.7024143839193153,1.3512071919596578]***

### Fourth-Order Yoshida Symplectic Method

The fourth-order Yoshida method is constructed as a composition of second-order symplectic integrators:

$$
\Phi^{(4)}(h)
=
\Phi^{(2)}(w_1 h)
\circ
\Phi^{(2)}(w_0 h)
\circ
\Phi^{(2)}(w_1 h),
$$

where the coefficients are given by:

$$
w_1 = \frac{1}{2 - 2^{1/3}},
\qquad
w_0 = -\frac{2^{1/3}}{2 - 2^{1/3}}.
$$

This scheme preserves the symplectic structure of the system and has a global error of order:

$$
\mathcal{O}(h^4).
$$

### Yoshida Coefficients (Symmetric Composition)

To construct an integrator of order $2n+2$ from one of order $2n$, the scale factors $x_1$ and $x_0$ defined by Haruo Yoshida are used:

$$x_1 = \frac{1}{2 - 2^{1/(2n+1)}}$$

$$x_0 = \frac{-2^{1/(2n+1)}}{2 - 2^{1/(2n+1)}}$$

#### Consistency and Order Conditions
These coefficients must satisfy the following equalities to guarantee accuracy and time conservation:

1. **Sum of steps:** $x_0 + 2x_1 = 1$
2. **Error cancellation:** $x_0^{2n+1} + 2x_1^{2n+1} = 0$

#### Example for 4th Order ($n=1$)
Substituting $n=1$ into the general formulas, we obtain the values used in the Forest-Ruth/Neri integrator:

$$x_1 = \frac{1}{2 - 2^{1/3}} \approx 1.35120719195966$$

$$x_0 = \frac{-2^{1/3}}{2 - 2^{1/3}} \approx -1.70241438391932$$

In [19]:
degree = 12  # You can change to 4, 6, 8, 10, 12...
c_yoshida, d_yoshida = fm.get_yoshida_coefficients(degree)
print("c =", c_yoshida)
print("d =", d_yoshida)

c = [ 1.0298920574908217  -0.2676906248614055  -0.2676906248614055
 -0.1531432547733972   0.307495780426425    0.307495780426425
 -0.1531432547733972  -0.2676906248614055  -0.2676906248614055
 -0.10720096340074144  0.29555441181826814  0.29555441181826814
  0.16908386168518483 -0.33950286666776086 -0.33950286666776086
  0.16908386168518483  0.29555441181826814  0.29555441181826814
 -0.10720096340074144 -0.2676906248614055  -0.2676906248614055
 -0.1531432547733972   0.307495780426425    0.307495780426425
 -0.1531432547733972  -0.2676906248614055  -0.2676906248614055
 -0.08245288920997516  0.28912186639172793  0.28912186639172793
  0.1654038637636733  -0.3321138123178505  -0.3321138123178505
  0.1654038637636733   0.28912186639172793  0.28912186639172793
  0.11578344453960845 -0.3192164208569078  -0.3192164208569078
 -0.18262067150260353  0.3666833775263713   0.3666833775263713
 -0.18262067150260353 -0.3192164208569078  -0.3192164208569078
  0.11578344453960845  0.28912186639172793  0.28

**VARIABLE CREATION**

In [20]:
# SIMULATION TIME
t = 10_000  # years
simulations = int((t/dt_const))
print("Iterations:", simulations)

r_container = np.zeros((simulations+1, 10 , 3), dtype=user_float)
p_container = np.zeros((simulations+1, 10 , 3), dtype=user_float)

# Initial values assignment
r_container[0, ...] = position_t0
p_container[0, ...] = momentum_t0

print(r_container.shape)

Iterations: 20000000
(20000001, 10, 3)


In [21]:
for i in range(simulations):
    r_container[i+1, ...], p_container[i+1, ...] = fm.Yoshida(r_container[i, ...], p_container[i, ...], mass, G_const, dt_const, coef_c=c_yoshida, coef_d=d_yoshida)
    if i % 1000 == 0:
        clear_output(wait=True)
        print(f"Progress: {((i+1)/(simulations))*100:.4f}%")
        
clear_output(wait=True)

print(f"Progress: 100%", "Success")


Progress: 100% Success


In [ ]:
np.savez("../results/positions.npz", pos=r_container)
np.savez("../results/momenta.npz", mon=p_container)

print("Saved successfully")

Saved successfully
